[Guide Index](README.md)


# 06b Working with Backend Graph Databases

Most examples in the StarLayer user guide demonstrate the functionality of `StarLayerGraph` using the in-memory graph. Typically `StarLayerGraph` will be used with backend datastores to persist data.

`StarLayerGraph` provides two ways to persist data using a backend graph store:
- **A SPARQL HTTP endpoint**.
- **A graph store**

The examples below demonstrate how to configure `StarLayerGraph` with backend stores.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later cells reuse variables from earlier ones.

Section 1 needs a running Oxigraph and/or Fuseki instance (each example skips cleanly if its endpoint isn't reachable) — every example clears all existing data in the dataset it connects to (`CLEAR ALL` / equivalent), so don't point one at a database you care about. Section 2 needs the `rdflib-sqlalchemy` package, installed inline by that section's own first cell.


In [1]:
from starlayer.graph import ENTAILMENT, StarLayerGraph, Namespace, TripleTerm

EX = Namespace("http://example.org/")

## 1. Using a SPARQL endpoint

`StarLayerGraph` can work against any SPARQL endpoint via a `SPARQLUpdateStore`.  `StarLayerGraph` has been tested against both Oxigraph and Fuseki.  Both support RDF 1.2 and native RDF 1.1 modes.  

RDF 1.2 mode is a better choice when working with RDF 1.2 graphs. RDF 1.1 mode is useful when integrating RDF 1.2 data with an existing SPARQL graph that operates using RDF 1.1.

The following examples require a running instance of Oxigraph and/or Fuseki.  Initiating a session is outside the scope of this user guide.

The examples assume the following end-points.  Your end-points may differ.
- Oxigraph: `http://localhost:7878/`
- Fuseki: `http://localhost:3030/starlayergraph` (5.5+ needed for native RDF 1.2)


In [2]:

import urllib.error
import requests
from rdflib.plugins.stores.sparqlstore import SPARQLUpdateStore

def demo_sparql_endpoint(label, query_endpoint, update_endpoint, auth=None, backend="rdf-1.2"):
    """Connect, clear, write a reification, and print the resulting graph."""
    def make_store():
        return SPARQLUpdateStore(query_endpoint=query_endpoint, update_endpoint=update_endpoint, auth=auth)

    try:
        g = StarLayerGraph(store=make_store(), identifier=EX.main, backend=backend)
        g.update("CLEAR ALL")
    except (urllib.error.URLError, requests.exceptions.ConnectionError):
        # backend="rdf-1.1" goes through plain rdflib's own SPARQLUpdateStore,
        # which raises urllib.error.URLError when unreachable; backend="rdf-1.2"
        # (native) goes through starlayer's own HTTP layer, which uses
        # requests directly (backends/native.py) and raises
        # requests.exceptions.ConnectionError instead - both are caught here
        # so this guard actually protects both call shapes below, not just one.
        print(f"{label}: skipping - no live instance at {query_endpoint}")
        return

    g.bind("ex", EX)
    g.add_reification(EX.claim, TripleTerm(EX.bob, EX.knows, EX.carol))
    print(f"{label}:")
    print(g.serialize())

# test oxigraph
demo_sparql_endpoint(
    "Oxigraph",
    query_endpoint="http://localhost:7878/query",
    update_endpoint="http://localhost:7878/update",
)

# test fueski
demo_sparql_endpoint(
    "Fuseki",
    query_endpoint="http://localhost:3030/starlayergraph/query",
    update_endpoint="http://localhost:3030/starlayergraph/update",
    auth=("admin", "admin"),
)

Oxigraph:
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .

Fuseki:
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .



### 1.1 RDF 1.1 SPARQL endpoint

When operating with an RDF 1.1 SPARQL end-point, change the backend to "rdf-1.1".  

Note that RDF 1.1 mode handles RDF 1.2 terms, such as triple terms.

In [3]:
demo_sparql_endpoint(
    "Oxigraph, rdf-1.1 mode",
    query_endpoint="http://localhost:7878/query",
    update_endpoint="http://localhost:7878/update",
    backend="rdf-1.1",
)

Oxigraph, rdf-1.1 mode:
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .



### 1.2 Reasoning-enabled Fuseki (`entailment="native"`)

`query(..., entailment="native")` (only legal with `backend="rdf-1.2"`) changes nothing about how the query is sent - it's a self-documenting label, since the native path already passes queries straight through unmodified. Entailment support is entirely a property of how the *endpoint itself* is configured. See [fuseki-reasoning-setup.md](../../packages/starlayer/docs/graph/fuseki-reasoning-setup.md) for how to stand up a Fuseki instance that reasons before answering - this example expects one at `localhost:3031`, a dataset named `starlayergraph-rdfs`, separate from the plain one above, started with `--rdfs=schema.ttl` declaring `ex:Manager rdfs:subClassOf ex:Employee`. See the [Graph Inferencing guide](02b-graphs-inferencing.ipynb) for `entailment=ENTAILMENT.RDFS`/`ENTAILMENT['OWL-RDF-Based']` - `StarLayerGraph`'s own reasoning, which needs none of this setup.

In [4]:
try:
    store = SPARQLUpdateStore(
        query_endpoint="http://localhost:3031/starlayergraph-rdfs/query",
        update_endpoint="http://localhost:3031/starlayergraph-rdfs/update",
    )
    g_rdfs = StarLayerGraph(store=store, backend="rdf-1.2")
    g_rdfs.update("CLEAR ALL")
except (urllib.error.URLError, requests.exceptions.ConnectionError):
    print("skipping - no live reasoning-enabled instance at localhost:3031")
else:
    g_rdfs.bind("ex", EX)
    g_rdfs.update("PREFIX ex: <http://example.org/> INSERT DATA { ex:alice a ex:Manager }")

    rows = list(g_rdfs.query(
        "PREFIX ex: <http://example.org/> SELECT ?x WHERE { ?x a ex:Employee }",
        entailment="native",
    ))
    print("alice is an Employee (via rdfs:subClassOf, reasoned by Fuseki itself):", [str(r.x) for r in rows])

skipping - no live reasoning-enabled instance at localhost:3031


## 2. Using a graph store

Any rdflib `Store` plugin works transparently with `StarLayerGraph`. 

In [5]:
!pip install -q rdflib-sqlalchemy

import tempfile
import warnings

with warnings.catch_warnings():
    # rdflib-sqlalchemy's own import warns about its (still-working) use of
    # the deprecated pkg_resources API - not something this guide is concerned with.
    warnings.filterwarnings("ignore", category=UserWarning, module="rdflib_sqlalchemy")
    import rdflib_sqlalchemy
rdflib_sqlalchemy.registerplugins()

db_path = tempfile.mktemp(suffix=".sqlite")
uri = f"sqlite:///{db_path}"

writer = StarLayerGraph(store="SQLAlchemy", identifier=EX.main)
writer.open(uri, create=True)
writer.bind("ex", EX)
writer.add_reification(EX.claim, TripleTerm(EX.bob, EX.knows, EX.carol))
writer.commit()
writer.close()

# fresh graph object, same database file - proves the data actually persisted
reader = StarLayerGraph(store="SQLAlchemy", identifier=EX.main)
reader.open(uri, create=False)
reader.bind("ex", EX)
print(reader.serialize())
reader.close()

@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim rdf:reifies <<( ex:bob ex:knows ex:carol )>> .

